# Error Per Layered Gate (EPLG)

In [ ]:
%load_ext autoreload
%autoreload 2
from iqm.benchmarks.logging_config import configure_logging
configure_logging()

## Set IQM Token if using Resonance.

In [ ]:
import os
from iqm.qiskit_iqm import IQMProvider

token = ""
os.environ["IQM_TOKEN"] = token
quantum_computer = "" # provide actual quantum computer name. i.e. "emerald", "garnet", "sirius"
iqm_server_url = "https://resonance.iqm.tech/" # provide actual IQM server URL
os.environ["IQM_SERVER_URL"] = iqm_server_url

provider = IQMProvider(iqm_server_url, quantum_computer=quantum_computer)
backend = provider.get_backend()

## Randomized Benchmarking Configuration

In [ ]:
from iqm.benchmarks.randomized_benchmarking.eplg.eplg import EPLGBenchmark, EPLGConfiguration
import matplotlib.pyplot as plt
import numpy as np

EPLG measures the layer fidelity of `n - 1` two-qubit gates connecting `n` qubits, split into disjoint layers
that are each benchmarked by parallel direct RB.

Without `custom_qubits_array`, the edges are selected for you. Selections are scored by their layer fidelity: the
fidelities of their edges (by default the calibrated 2Q Clifford fidelities, or CZ fidelities if the calibration
has none), plus a single-qubit Clifford and T1/T2 idling for every qubit without an edge in a layer.

* `edge_selection="tree"` (default): the best chain (2 layers) compared with the best trees needing 3 or 4 layers.
* `edge_selection="chain"`: the best chain, found by a depth-first search over the chains.
* `search_budget` bounds that search (default `10**6` steps); when it is reached, the best chain found is used.
* `edge_fidelities` scores the edges with your own fidelities, used as given; only those edges can be selected.
* `edge_selection_rank` runs the second (or n-th) best of the logged ranking instead of the best.

In [ ]:
EXAMPLE_EPLG = EPLGConfiguration(
    drb_depths=sorted(list(set(np.geomspace(1, 50, num=4, endpoint=True, dtype=int).tolist())), reverse=True),
    drb_circuit_samples=25,
    shots=2**8,
    max_circuits_per_batch=100,
    # Number of qubits to select edges for; the selection has num_qubits - 1 edges.
    num_qubits=18,
    edge_selection="tree",
    # Exact number of disjoint layers (default: the fewest possible). More layers run fewer gates at a time.
    # num_disjoint_layers=2,
    # Qubits every edge selection must avoid, by index or name:
    # excluded_qubits=["QB5"],
    # Score the edges with other fidelities than the calibrated 2Q Clifford ones, e.g. the CZ fidelities:
    # edge_fidelities=edge_fidelities_from_calibration(backend, "cz_gate_fidelity"),
    # Budget of the chain search; when reached, the best chain found so far is used and a warning is logged.
    # search_budget=10**6,
    # Run another of the logged ranked selections, 0 being the best:
    edge_selection_rank=0,
    # Or skip the selection and give the edges yourself, as a flat list (split into the fewest layers) ...
    # custom_qubits_array=[(0, 1), (1, 2), (2, 3)],
    # ... or as explicit layers of disjoint edges, characterised exactly as given:
    # custom_qubits_array=[[(0, 1), (2, 3)], [(1, 2)]],
)

## Run the experiment

In [ ]:
benchmark_eplg = EPLGBenchmark(backend, EXAMPLE_EPLG)
run_eplg = benchmark_eplg.run()

## Perform the analysis

In [ ]:
plt.rcParams['figure.dpi'] = 175

In [ ]:
result_eplg = benchmark_eplg.analyze()

## Inspect all the fidelities and the final EPLG

In [ ]:
for o in result_eplg.observations:
    if o.name != "eplg":
        print(f"{o.name} {o.identifier.custom_identifier}: {100*o.value:.2f} +/- {100*o.uncertainty:.2f} %")
    else:
        print(f"\n{o.name} {o.identifier.custom_identifier}: {o.value:.2e} +/- {o.uncertainty:.2e}")

## Plot a visualization of the selected layers and DRB decays

Show all available plots

In [ ]:
result_eplg.plots.keys()

Show the two layered fidelities graphs (colored by fidelity and labelled with the layer, then colored by layer and
labelled with the fidelity) and a few DRB decay curves

In [ ]:
drb_pairs = list(result_eplg.plots.keys())[-3:] # or custom
main_plots = list(result_eplg.plots.keys())[:2]
from IPython.display import display
for k in main_plots + drb_pairs:
    display(result_eplg.plots[k])